# System Identification and Inverse Filtering

Estimate a channel response from a clean/distorted pair and restore another recording with a regularized inverse filter.

Run the cells in order. This notebook is independent of the other notebooks.


In [ ]:
from pathlib import Path

# Resolve paths when launched from the repository root or notebooks directory.
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA_DIR = ROOT / "data" / "audio"
OUTPUT_DIR = ROOT / "outputs"
if not DATA_DIR.is_dir():
    raise FileNotFoundError("Start Jupyter from the repository root or notebooks directory.")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## Frequency-response estimation


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.io import wavfile


# ---------- audio utils ----------
def mono_float(x: np.ndarray) -> np.ndarray:
    """Convert to mono float32 in [-1, 1] (mean over channels if stereo)."""
    if x.ndim == 2:
        x = x.mean(axis=1)
    if np.issubdtype(x.dtype, np.integer):
        x = x.astype(np.float32) / np.iinfo(x.dtype).max
    else:
        x = x.astype(np.float32)
    return x


def read_wav_mono(path: Path) -> tuple[int, np.ndarray]:
    """Read WAV at 'path' and return (fs, mono_float_signal)."""
    fs, x = wavfile.read(str(path))
    return fs, mono_float(x)


def ensure_same_sr(tagged: list[tuple[str, int]]) -> int:
    """Ensure all sampling rates match; return the common fs."""
    tags, fss = zip(*tagged)
    if len(set(fss)) != 1:
        msg = "Sampling rate mismatch: " + ", ".join(f"{t}={fs} Hz" for t, fs in tagged)
        raise ValueError(msg)
    return fss[0]


def crop_pair_to_shorter(a: np.ndarray, b: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Crop two arrays to the same (shorter) length."""
    L = min(len(a), len(b))
    return a[:L], b[:L]


def next_pow2(n: int) -> int:
    """Small helper for next power of two."""
    return int(2 ** np.ceil(np.log2(max(1, n))))


def fft_size_for_signals(*signals: np.ndarray) -> int:
    """Choose FFT size as next power-of-two ≥ max length across signals."""
    longest = max(len(s) for s in signals if s is not None)
    return next_pow2(longest)


# ---------- main workflow ----------
def load_inputs(clean_path=(DATA_DIR / "clean1.wav"),
                distorted1_path=(DATA_DIR / "distorted1.wav"),
                distorted2_path=(DATA_DIR / "distorted2.wav")):
    clean_path = Path(clean_path)
    d1_path = Path(distorted1_path)
    d2_path = Path(distorted2_path)

    fs_c, clean1 = read_wav_mono(clean_path)
    fs_d1, distorted1 = read_wav_mono(d1_path)
    fs_d2, distorted2 = read_wav_mono(d2_path)

    fs = ensure_same_sr([("clean1", fs_c), ("distorted1", fs_d1), ("distorted2", fs_d2)])

    clean1, distorted1 = crop_pair_to_shorter(clean1, distorted1)

    N = fft_size_for_signals(clean1, distorted1, distorted2)

    print(f"FFT size: {N} | fs: {fs}")
    return fs, clean1, distorted1, distorted2, N


fs, clean1, distorted1, distorted2, N = load_inputs()

print("\n")

import numpy as np
import matplotlib.pyplot as plt
from typing import Tuple, Dict, Optional


# ---------- frequency / FFT helpers ----------
def freq_axis(N: int, fs: float, *, shift: bool = True) -> np.ndarray:
    """Return frequency axis in Hz (fftshifted by default)."""
    f = np.fft.fftfreq(N, d=1.0/fs)
    return np.fft.fftshift(f) if shift else f


def fft_with_len(x: np.ndarray, N: int, window: Optional[str] = None) -> np.ndarray:
    """Compute FFT of x with zero-padding/truncation to N, optional windowing."""
    xN = x if len(x) == N else np.pad(x, (0, max(0, N - len(x))))[:N]
    if window:
        w = get_window(window, N).astype(xN.dtype, copy=False)
        xN = xN * w
    return np.fft.fft(xN, n=N)


def get_window(name: str, N: int) -> np.ndarray:
    """Minimal window provider; extend as needed."""
    name = name.lower()
    if name in ("hann", "hanning"):
        n = np.arange(N)
        return 0.5 - 0.5 * np.cos(2 * np.pi * n / (N - 1))
    if name in ("hamming",):
        n = np.arange(N)
        return 0.54 - 0.46 * np.cos(2 * np.pi * n / (N - 1))
    # default rectangular
    return np.ones(N)


# ---------- FRF estimation ----------
def estimate_frf(
    x: np.ndarray,
    y: np.ndarray,
    *,
    fs: float,
    N: int,
    eps: float = 1e-8,
    window: Optional[str] = None
) -> Dict[str, np.ndarray]:
    """
    Estimate frequency response H = Y/X with soft regularization (eps) to avoid divide-by-zero.
    Returns dict with f (shifted), H, |H|, angle(H) unwrapped, and dB magnitude (shifted).
    """
    X = fft_with_len(x, N, window=window)
    Y = fft_with_len(y, N, window=window)

    # Safe division: if |X| very small, add eps in the phase of X
    H = Y / (X + eps)

    # Unshifted metrics
    Hmag = np.abs(H)
    Hpha = np.unwrap(np.angle(H))

    # Shift for plotting
    f_plot = freq_axis(N, fs, shift=True)
    Hmag_db_plot = 20 * np.log10(np.fft.fftshift(Hmag) + 1e-12)  # floor to avoid -inf
    Hpha_plot = np.fft.fftshift(Hpha)

    return {
        "H": H,
        "Hmag": Hmag,
        "Hpha": Hpha,
        "f_plot": f_plot,
        "Hmag_db_plot": Hmag_db_plot,
        "Hpha_plot": Hpha_plot,
    }


# ---------- plotting ----------
def plot_frf(
    f: np.ndarray,
    mag_db: np.ndarray,
    pha: np.ndarray,
    *,
    title_prefix: str = "Estimated H(f)"
) -> None:
    """Plot magnitude (dB) and unwrapped phase."""
    plt.figure(figsize=(10, 3))
    plt.plot(f, mag_db, color = "red")
    plt.title(f"{title_prefix} — |H| (dB)")
    plt.xlabel("frequency (Hz)")
    plt.ylabel("|H| (dB)")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    print()
    plt.figure(figsize=(10, 3))
    plt.plot(f, pha, color = "green")
    plt.title(f"{title_prefix} — ∠H (unwrapped)")
    plt.xlabel("frequency (Hz)")
    plt.ylabel("phase (rad)")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

res = estimate_frf(clean1, distorted1, fs=fs, N=N, eps=1e-8, window=None)
plot_frf(res["f_plot"], res["Hmag_db_plot"], res["Hpha_plot"], title_prefix="H1(f)")

## Regularized inverse filtering


In [ ]:
import numpy as np
from pathlib import Path
from scipy.io import wavfile
from typing import Optional, Tuple


# ---------- FRF estimation ----------
def estimate_regularized_frf(x: np.ndarray, y: np.ndarray, N: int, alpha: float = 1e-12) -> np.ndarray:
    """
    Estimate frequency response H(f) = Y/X with small regularization.
    """
    X = np.fft.fft(x, n=N)
    Y = np.fft.fft(y, n=N)
    return (Y * np.conj(X)) / (np.abs(X) ** 2 + alpha)


# ---------- inverse FRF helpers ----------
def tikhonov_inverse_frf(H: np.ndarray, alpha: float, eps: float = 1e-12) -> np.ndarray:
    """
    Tikhonov-regularized inverse filter.
      H_inv[k] = H*[k] / (|H[k]|^2 + alpha)
    """
    denom = (np.abs(H) ** 2) + max(alpha, eps)
    return np.conj(H) / denom


def apply_inverse_filter_freq(
    y: np.ndarray,
    H_inv: np.ndarray,
    N: int,
    out_len: Optional[int] = None
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Apply inverse filter in frequency domain.
    """
    Y = np.fft.fft(y, n=N)
    X_hat = Y * H_inv
    x_hat_full = np.fft.ifft(X_hat)
    if out_len is None:
        out_len = len(y)
    x_hat = np.real(x_hat_full[:out_len])
    return X_hat, x_hat


# ---------- I/O ----------
def save_wav_int16(path: Path, fs: int, x: np.ndarray) -> None:
    """
    Save float signal in [-1,1] to 16-bit PCM WAV.
    """
    x = np.clip(x, -1.0, 1.0)
    wav = (x * 32767.0).astype(np.int16)
    wavfile.write(str(path), fs, wav)


alpha = 1e-3

# Step 1: estimate FRF H
H = estimate_regularized_frf(clean1, distorted1, N)

# Step 2: build inverse FRF
H_inv = tikhonov_inverse_frf(H, alpha)

# Step 3: apply to distorted2
_, x2_hat = apply_inverse_filter_freq(distorted2, H_inv, N, out_len=len(distorted2))

# Step 4: save result
out_dir = OUTPUT_DIR
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "recovered2.wav"
save_wav_int16(out_path, fs, x2_hat)

## Spectrum and RMS comparison


In [ ]:
def rms(x):
  return np.sqrt(np.mean(x**2) + 1e-12)

print("RMS:")
print("distorted2:", rms(distorted2), " | recovered2:", rms(x2_hat))

print()

plt.figure(figsize=(8,3))
plt.magnitude_spectrum(distorted2, Fs=fs, color = "red")
plt.title("distorted2 -> Magnitude spectrum")
plt.grid(True, alpha=0.3)
plt.show()

print()

plt.figure(figsize=(8,3))
plt.magnitude_spectrum(x2_hat, Fs=fs, color = "green")
plt.title("recovered2 -> Magnitude spectrum")
plt.grid(True, alpha=0.3)
plt.show()